In [1]:
# imports
import random
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import keras

In [2]:
# Change pixel values to be range [0,1]
def normalize_img(image, label):
	"""Normalizes images: `uint8` -> `float32`."""
	return tf.cast(image, tf.float32) / 255., label

In [3]:
# load data
ds = tfds.load('pneumonia_mnist', split='train[:4700]', shuffle_files=False, as_supervised=True)
ds = ds.map(normalize_img, num_parallel_calls=tf.data.AUTOTUNE)

In [4]:
# generate randomly sampled dataset for training target model
zeros = np.zeros(int(len(ds)/2))
ones = np.ones(int(len(ds)/2))
				 

filter = np.concatenate((zeros, ones))
random.shuffle(filter)

filtered_features = []
filtered_labels = []

for step, (x, y) in enumerate(ds):
	if filter[step] == 1:
		filtered_features.append(x)
		filtered_labels.append(y)

filtered_ds = tf.data.Dataset.from_tensor_slices((filtered_features, filtered_labels))

In [6]:
subsets = [[] for i in range(10)]
features = [[] for i in range(10)]
labels = [[] for i in range(10)]

for step, (x, y) in enumerate(ds):
	for offset in range(5):
		index = (step + offset) % 10
		features[index].append(x)
		labels[index].append(y)

subsets = [tf.data.Dataset.from_tensor_slices((features[i], labels[i])) for i in range(len(features))]

In [7]:
# cache and prefetch datasets for faster performance
ds = ds.cache()
ds = ds.batch(64)
ds = ds.prefetch(tf.data.AUTOTUNE)

filtered_ds = filtered_ds.cache()
filtered_ds = filtered_ds.batch(64)
filtered_ds = filtered_ds.prefetch(tf.data.AUTOTUNE)


for i in range(len(subsets)):
	subsets[i] = subsets[i].cache()
	subsets[i] = subsets[i].batch(64)
	subsets[i] = subsets[i].prefetch(tf.data.AUTOTUNE)

In [8]:
# target model and shadow model architecture
def get_classifier():
	model = tf.keras.models.Sequential([
	tf.keras.layers.Conv2D(128, (3, 3), activation="relu", input_shape=(28, 28, 1)),
	tf.keras.layers.MaxPool2D(2,2),
	tf.keras.layers.Conv2D(128, (3, 3), activation="relu"),
	tf.keras.layers.MaxPool2D(2,2),
	tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
	tf.keras.layers.MaxPool2D(2,2),
	tf.keras.layers.Flatten(),
	tf.keras.layers.Dense(64, activation='relu'),
	tf.keras.layers.Dense(2)
	])

	return model

In [9]:
# train target model on in data
target_model = get_classifier()

target_model.compile(
optimizer=tf.keras.optimizers.Adam(0.001),
loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
metrics=[tf.keras.metrics.SparseCategoricalAccuracy()],
)

target_model.fit(
    filtered_ds,
    epochs=25,
)

C:\Users\Skylar Marosi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - loss: 0.5729 - sparse_categorical_accuracy: 0.7404
Epoch 2/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 0.4901 - sparse_categorical_accuracy: 0.7477
Epoch 3/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.3673 - sparse_categorical_accuracy: 0.8409
Epoch 4/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.2814 - sparse_categorical_accuracy: 0.8826
Epoch 5/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.2289 - sparse_categorical_accuracy: 0.9068
Epoch 6/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.2018 - sparse_categorical_accuracy: 0.9166
Epoch 7/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 0.1937 - sparse_categorical_accuracy: 0.9200
Epoch 8/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.1804 - sparse_categorical_accuracy: 0.9247
Epoch 9/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 0.1655 - sparse_categorical_accuracy: 0.9336
Epoch 10/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.15

In [ ]:
# train and save imitative models

mse = keras.losses.MeanSquaredError()
ce = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

for i in range(10):
	optimizer = tf.keras.optimizers.Adam(0.001)
	im_model = get_classifier()

	warmup = True

	if warmup:
		# warmup model with out data
		for epoch in range(50):
			subsets[i] = subsets[i].shuffle(buffer_size=10)
			for step, (x_batch_out, y_batch_out) in enumerate(subsets[i]):
				with tf.GradientTape() as tape:
					student_logits = im_model(x_batch_out, training=False)

					loss_value = ce(y_batch_out, student_logits)
					
				grads = tape.gradient(loss_value, im_model.trainable_weights)

				optimizer.apply_gradients(zip(grads, im_model.trainable_weights))


	# next, train model with mixture of regular and imitative loss
	best_loss = 1
	for epoch in range(80):
		subsets[i] = subsets[i].shuffle(buffer_size=10)
		for step, (x_batch_out, y_batch_out) in enumerate(subsets[i]):
			with tf.GradientTape() as tape:
				student_logits = im_model(x_batch_out, training=False) / 2

				teacher_logits = target_model(x_batch_out) / 2

				student_probas = tf.nn.softmax(student_logits)

				teacher_probas = tf.nn.softmax(teacher_logits)

				# loss measuring accuracy of this out-model
				regular_loss = ce(y_batch_out, student_probas)

				# loss measuring how closely this out-model follows the target model
				imitate_loss = mse(teacher_probas, student_probas)

				# save best model
				if (imitate_loss < best_loss):
					best_loss = imitate_loss
					im_model.save_weights(f'./imitative_models/im_model_{i+1}.weights.h5')

				# both losses are used to train the model
				loss_value = 0.5 * regular_loss + 0.5 * imitate_loss
				
			grads = tape.gradient(loss_value, im_model.trainable_weights)

			optimizer.apply_gradients(zip(grads, im_model.trainable_weights))
			
	print(f"finished training model {i+1}")



tf.Tensor(
[[0.5004517  0.49954838]
 [0.50112665 0.4988733 ]
 [0.5016387  0.49836132]
 [0.5012277  0.49877232]
 [0.50091845 0.49908155]
 [0.5016897  0.4983103 ]
 [0.50093573 0.4990642 ]
 [0.5010054  0.49899462]
 [0.5010076  0.49899238]
 [0.5013636  0.49863642]
 [0.50088966 0.49911037]
 [0.5005485  0.4994515 ]
 [0.50118965 0.49881026]
 [0.50008667 0.49991333]
 [0.50110525 0.49889472]
 [0.5010534  0.49894664]
 [0.50184536 0.49815467]
 [0.50160015 0.49839982]
 [0.50103265 0.49896735]
 [0.5013903  0.49860972]
 [0.5016729  0.49832714]
 [0.50098467 0.4990153 ]
 [0.50120544 0.49879462]
 [0.50085104 0.4991489 ]
 [0.5008582  0.4991418 ]
 [0.501146   0.49885398]
 [0.5005838  0.49941608]
 [0.50042397 0.49957603]
 [0.50031215 0.49968785]
 [0.50133604 0.49866393]
 [0.5010718  0.49892813]
 [0.5007453  0.4992547 ]
 [0.5009123  0.49908772]
 [0.50039375 0.4996062 ]
 [0.50157124 0.4984288 ]
 [0.5006411  0.49935895]
 [0.5004215  0.4995785 ]
 [0.5009238  0.4990762 ]
 [0.5009224  0.49907753]
 [0.5016789  0

KeyboardInterrupt: 

In [11]:
#load imitative models for inference
im_models = []
for i in range(10):
	im_model = get_classifier()
	im_model.load_weights(f'./imitative_models/im_model_{i+1}.weights.h5')
	im_models.append(im_model)

In [12]:
#calculate scaled confidence score for a query
def scaled_confidence_score(x, y, model):
	pred = model.predict(x)

	confidences = []
	for i in range(len(x)):
		probas = tf.nn.softmax(pred[i])
		real = probas[y[i]] + 1e-45
		other = probas[1 - y[i]] + 1e-45
		phi = np.log10(real) - np.log10(other)
		confidences.append(phi)

	return confidences

In [13]:
correct = 0
count = 0
for step, (x_batch, y_batch) in enumerate(ds):
	target_phi = scaled_confidence_score(x_batch, y_batch, target_model)

	in_indexes = []
	for offset in range(5):
		index = (step + offset) % 10
		in_indexes.append(index)

	mean_in_phi = np.zeros(len(x_batch))
	for i in range(len(im_models)):
		if i in in_indexes:
			mean_in_phi = mean_in_phi + scaled_confidence_score(x_batch, y_batch, im_models[i])
	mean_in_phi = mean_in_phi / (len(im_models)/2)

	mean_out_phi = np.zeros(len(x_batch))
	for i in range(len(im_models)):
		if i not in in_indexes:
			mean_out_phi = mean_out_phi + scaled_confidence_score(x_batch, y_batch, im_models[i])
	mean_out_phi = mean_out_phi / (len(im_models)/2)

	out_dist = (target_phi - mean_out_phi)**2
	in_dist = (target_phi - mean_in_phi)**2

	for i in range(len(x_batch)):
		index = step * 64 + i
		membership_guess = 0
		if (in_dist[i] < out_dist[i]):
			membership_guess = 1

		if filter[index] == membership_guess:
			correct += 1
		count += 1

advantage = correct / count

print(f'Advantage is {advantage}')

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
2/2 ━━━━━━━━